# Backstage-Container in Kubernetes starten

Minimaler Ablauf: Namespace erstellen, Secrets anlegen, Image als Deployment starten und über einen Service erreichbar machen.

## 1. Variablen festlegen

Zuerst werden Namespace, Image und die benötigten Zugangsdaten zentral definiert. Passe insbesondere den Image-Namen an deine Registry an.

In [ ]:
import os

os.environ['BACKSTAGE_NAMESPACE'] = 'backstage'
os.environ['BACKSTAGE_IMAGE'] = 'registry.example.com/mybackstage:1.0.0'

os.environ['MICROSOFT_CLIENT_ID'] = 'deine-client-id'
os.environ['MICROSOFT_CLIENT_SECRET'] = 'dein-client-secret'
os.environ['AZURE_TENANT_ID'] = 'deine-tenant-id'
os.environ['GITHUB_TOKEN'] = 'dein-github-token'
os.environ['GOOGLE_CLIENT_ID'] = 'deine-google-client-id'
os.environ['GOOGLE_CLIENT_SECRET'] = 'dein-google-client-secret'

## 2. Namespace erstellen

Die Backstage-Ressourcen werden in einem eigenen Kubernetes Namespace abgelegt.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Namespace
metadata:
  name: ${BACKSTAGE_NAMESPACE}
EOF

## 3. Secrets in Kubernetes anlegen

Die Zugangsdaten werden als Kubernetes Secret gespeichert und später als Umgebungsvariablen in den Container geladen.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Secret
metadata:
  name: backstage-secrets
  namespace: ${BACKSTAGE_NAMESPACE}
type: Opaque
stringData:
  MICROSOFT_CLIENT_ID: "${MICROSOFT_CLIENT_ID}"
  MICROSOFT_CLIENT_SECRET: "${MICROSOFT_CLIENT_SECRET}"
  AZURE_TENANT_ID: "${AZURE_TENANT_ID}"
  GITHUB_TOKEN: "${GITHUB_TOKEN}"
  GOOGLE_CLIENT_ID: "${GOOGLE_CLIENT_ID}"
  GOOGLE_CLIENT_SECRET: "${GOOGLE_CLIENT_SECRET}"
EOF

## 4. Backstage Deployment erstellen

Das Deployment startet das vorbereitete Backstage-Image. Alle Werte aus `backstage-secrets` werden als Umgebungsvariablen übernommen.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: backstage
  namespace: ${BACKSTAGE_NAMESPACE}
spec:
  replicas: 1
  selector:
    matchLabels:
      app: backstage
  template:
    metadata:
      labels:
        app: backstage
    spec:
      containers:
        - name: backstage
          image: ${BACKSTAGE_IMAGE}
          imagePullPolicy: IfNotPresent
          ports:
            - name: http
              containerPort: 7007
          envFrom:
            - secretRef:
                name: backstage-secrets
EOF

## 5. Service erstellen

Der NodePort-Service veröffentlicht Backstage auf Port `30077` jedes Kubernetes-Nodes.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Service
metadata:
  name: backstage
  namespace: ${BACKSTAGE_NAMESPACE}
spec:
  type: NodePort
  selector:
    app: backstage
  ports:
    - name: http
      port: 7007
      targetPort: 7007
      nodePort: 30077
EOF

## 6. Backstage öffnen

Backstage ist anschliessend über die IP-Adresse eines Kubernetes-Nodes erreichbar:

`http://<NODE-IP>:30077`